# NB08 – Marktdynamik: Spread-Entwicklung & CAPEX-Lernkurve
### CAS Information Engineering – Scripting Project (Kür)
**Gruppe:** SC26_Gruppe_2 | **Datum:** März–Mai 2026

---
Analysiert die zwei wichtigsten **disruptiven Parameter** für den Business Case:

1. **Spread-Entwicklung** — steigt das Arbitrage-Potenzial strukturell?
2. **CAPEX-Lernkurve** — wann unterschreiten Batterien die Break-Even-Schwelle?

> ℹ️ Chart 7 (Spread-Zeitreihe) und Chart 8 (CAPEX×Spread-Heatmap) aus NB03  
> werden hier direkt **weiterverwendet und um Analyse-Kontext erweitert**.  
> Die zugrundeliegende `spread_zeitreihe.csv` wurde in NB02 Sektion 7 berechnet.

---
| [← NB07 Cross-Border](07_Cross_Border.ipynb) | [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB09 Revenue Stacking](09_Revenue_Stacking.ipynb) |
|:---|:---:|---:|


---
## 1. Daten laden

### Voraussetzungen
| Datei | Erzeugt in | Pfad | Zweck |
|-------|-----------|------|-------|
| `spread_zeitreihe.csv` | NB02 Sektion 7 | `data/intermediate/` | Monatlicher Spread, Volatilität, Negativpreise |
| `wirtschaftlichkeit.csv` | NB02 Sektion 4 | `data/intermediate/<szenario>/` | Break-Even-Basiswerte je Segment |

`spread_zeitreihe.csv` enthält bereits alle nötigen Kennzahlen:
- `spread_median`: Monatlicher Median des Intra-Tag-Spreads (p75−p25) [EUR/MWh]
- `volatility`: Monatliche Std.-Abweichung der Stundenpreise
- `neg_price_h`: Anzahl Stunden mit Negativpreisen pro Monat


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import os, warnings, json as _json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
warnings.filterwarnings('ignore')

with open('config.json') as _f:
    CFG = _json.load(_f)

MODE          = CFG['mode']
DIR_RAW       = os.path.join('data', 'raw')
DIR_PROCESSED = os.path.join('data', 'processed')
DIR_INTER     = os.path.join('data', 'intermediate')
SZ_AKTIV      = CFG['szenarien']['gleichzeitigkeit_aktiv']
DIR_INTER_SZ  = os.path.join(DIR_INTER, SZ_AKTIV)
CHARTS_DIR       = os.path.join('output', 'charts', SZ_AKTIV)
os.makedirs(CHARTS_DIR, exist_ok=True)
DPI = CFG['visualisierung']['output_dpi']  # SSOT: config.json

_viz=CFG.get('visualisierung',{}).get('farben',{})
BG_DARK  = _viz.get('bg_dark',  '#0d1117')
BG_PANEL = _viz.get('bg_panel', '#141414')
C_PRICE  = _viz.get('c_price',  '#FFA726')
C_LOAD   = _viz.get('c_load',   '#66BB6A')
C_CHARGE = _viz.get('c_charge', '#1565C0')
C_FEED   = _viz.get('c_feed',   '#B71C1C')
SEG_COLORS = _viz.get('seg_colors', ['#42A5F5','#66BB6A','#FFA726','#EF5350'])
# Marktdynamik-Parameter aus config.json
_md = CFG['kuer']['markt']
SPREAD_TRIGGER_EUR = _md['spread_breakeven_privat_eur_mwh']   # 30
CAPEX_LERNRATE     = _md['capex_lernrate_pct_pro_jahr']        # 10
CAPEX_TRIGGER      = _md['capex_ziel_privat_eur_kwh']          # 250

print(f'MODE           : {MODE}')
print(f'Spread-Trigger : > {SPREAD_TRIGGER_EUR} EUR/MWh Monatsmedian')
print(f'CAPEX-Lernrate : -{CAPEX_LERNRATE}%/Jahr')
print(f'CAPEX-Trigger  : < {CAPEX_TRIGGER} EUR/kWh installiert')

In [ ]:
# ── Daten laden ───────────────────────────────────────────────────────────────
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')
ECON_FILE   = os.path.join(DIR_INTER_SZ, 'wirtschaftlichkeit.csv')

df_sp   = None  # immer initialisieren
df_econ = None

if not os.path.exists(SPREAD_FILE):
    print('⚠️  spread_zeitreihe.csv fehlt → NB02 Sektion 7 ausführen.')
else:
    df_sp = pd.read_csv(SPREAD_FILE, parse_dates=['yearmonth'])

df_econ = pd.read_csv(ECON_FILE) if os.path.exists(ECON_FILE) else None

if df_sp is not None:
    import numpy as np
    print(f'Spread-Zeitreihe : {df_sp.shape} | {df_sp["yearmonth"].min().strftime("%Y-%m")} – {df_sp["yearmonth"].max().strftime("%Y-%m")}')
    print(f'Ø Spread         : {df_sp["spread_median"].mean():.1f} EUR/MWh')
    z = np.polyfit(range(len(df_sp)), df_sp['spread_median'], 1)
    print(f'Trend            : {z[0]:+.3f} EUR/MWh/Monat ({z[0]*12:+.2f} EUR/MWh/Jahr)')
    print(f'Negativpreis-Std : {df_sp["neg_price_h"].sum():,} total ({df_sp["neg_price_h"].mean():.1f}/Monat)')
    display(df_sp.head(3))
else:
    print('df_sp nicht verfügbar — nachfolgende Zellen überspringen.')


---
## 2. Analyse: Spread-Entwicklung

**Ausgangslage** (aus NB07_Erweiterungen Sektion 2, jetzt hier vertieft):

Der Intra-Tag-Spread ist der direkteste ROI-Treiber — jeder EUR/MWh mehr Spread  
bedeutet mehr Erlös pro Dispatch-Zyklus. 2023/2024 lagen die CH-Spreads historisch  
tief (normalisierte Post-Krisen-Preise nach dem Energiekrisen-Peak 2022).

Mit steigendem Anteil erneuerbarer Energien (Solar, Wind) steigt die strukturelle  
Volatilität: Mittags-Tief durch Solar, Abend-Spitze durch Nachfrage. Dieser Effekt  
ist in DE/AT bereits messbar und wird in der CH mit dem Solarausbau folgen.

### Break-Even-Tabelle nach Spread

| Spread [EUR/MWh] | Break-Even Privat (10kWh) | Bewertung |
|---|---|---|
| < 15 | > 30 Jahre | Nicht rentabel |
| 20–30 | 15–25 Jahre | Grenzwertig |
| 30–50 | 8–15 Jahre | Mit VPP attraktiv |
| > 50 | < 8 Jahre | Klar rentabel |

**Monitoring-Trigger:** Investition lohnt sich wenn CH-Monatsmedian-Spread  
**dauerhaft > 30 EUR/MWh** — Schwellenwert in `config.json → kuer.markt.spread_breakeven_privat_eur_mwh`.


In [ ]:
if df_sp is None:
    print('⚠️  Übersprungen — Spread-Daten fehlen.')
else:
    # ── Spread-Analyse: Trendberechnung & Monitoring ─────────────────────────────
    import numpy as np
    xn = np.arange(len(df_sp))
    z  = np.polyfit(xn, df_sp['spread_median'], 1)
    trend_j = z[0] * 12  # EUR/MWh pro Jahr
    
    # Extrapolation bis Trigger
    current = df_sp['spread_median'].mean()
    if trend_j > 0:
        jahre_bis_trigger = (SPREAD_TRIGGER_EUR - current) / trend_j
        print(f'Aktueller Ø Spread : {current:.1f} EUR/MWh')
        print(f'Trend              : {trend_j:+.2f} EUR/MWh/Jahr')
        print(f'Trigger ({SPREAD_TRIGGER_EUR} EUR/MWh): in ~{jahre_bis_trigger:.0f} Jahren (falls Trend hält)')
    else:
        print(f'Trend negativ ({trend_j:.2f}/Jahr) — Trigger unklar')
    
    # Monate über/unter Trigger
    über = (df_sp['spread_median'] >= SPREAD_TRIGGER_EUR).mean() * 100
    print(f'Monate >= {SPREAD_TRIGGER_EUR} EUR/MWh: {über:.0f}% der Datensätze')
    


---
## 3. Chart 08-A: Spread- & Volatilitäts-Zeitreihe

*Direkt übernommen aus NB03 Chart 7 — hier eingebettet für eigenständige Ausführbarkeit.*  
Trendlinie zeigt ob das Arbitrage-Potenzial strukturell steigt oder fällt.


In [ ]:
# ── Chart 7: Spread- & Volatilitäts-Zeitreihe ───────────────────────────────
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')  # aus NB02 Sektion 7

if not os.path.exists(SPREAD_FILE):
    print('spread_zeitreihe.csv nicht vorhanden → NB02 Sektion 6 ausführen.')
else:
    df_sp = pd.read_csv(SPREAD_FILE, parse_dates=['yearmonth'])
    x  = df_sp['yearmonth']
    xn = np.arange(len(df_sp))
    z  = np.polyfit(xn, df_sp['spread_median'], 1)

    # ── Gesamtchart (3 Panels) ────────────────────────────────────────────────
    fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
    fig.patch.set_facecolor('#0d1117')
    for ax in axes:
        ax.set_facecolor('#141414'); ax.tick_params(colors='#bbbbbb')
        for sp in ax.spines.values(): sp.set_edgecolor('#333333')
    fig.suptitle('Arbitrage-Potenzial Entwicklung — Monatlicher Spread & Volatilität',
                 color='white', fontsize=13, fontweight='bold')

    ax = axes[0]
    ax.fill_between(x, df_sp['spread_median'], alpha=0.3, color='#FFA726')
    ax.plot(x, df_sp['spread_median'], color='#FFA726', lw=2)
    ax.axhline(df_sp['spread_median'].mean(), color='white', lw=1, linestyle='--', alpha=0.5,
               label=f'Ø {df_sp["spread_median"].mean():.1f} EUR/MWh')
    ax.plot(x, np.polyval(z, xn), color='#EF5350', lw=1.5, linestyle=':',
            label=f'Trend ({z[0]:+.2f} EUR/MWh/Monat)')
    ax.set_ylabel('Spread [EUR/MWh]', color='#aaa')
    ax.set_title('Intra-Tag-Spread (p75−p25 Median)', color='white')
    ax.legend(fontsize=8, framealpha=0.3, facecolor='#111', labelcolor='white')
    ax.grid(True, alpha=0.10)

    ax = axes[1]
    ax.fill_between(x, df_sp['volatility'], alpha=0.3, color='#42A5F5')
    ax.plot(x, df_sp['volatility'], color='#42A5F5', lw=2)
    ax.axhline(df_sp['volatility'].mean(), color='white', lw=1, linestyle='--', alpha=0.5,
               label=f'Ø {df_sp["volatility"].mean():.1f} EUR/MWh')
    ax.set_ylabel('Volatilität [EUR/MWh]', color='#aaa')
    ax.set_title('Marktvolatilität (Std. Abweichung Stundenpreise)', color='white')
    ax.legend(fontsize=8, framealpha=0.3, facecolor='#111', labelcolor='white')
    ax.grid(True, alpha=0.10)

    ax = axes[2]
    colors_neg = ['#EF5350' if v > 0 else '#444' for v in df_sp['neg_price_h']]
    ax.bar(x, df_sp['neg_price_h'], color=colors_neg, alpha=0.85, width=25,
           label='Stunden mit Preis < 0 EUR/MWh')
    ax.set_ylabel('Negativpreis-Stunden', color='#aaa')
    ax.set_title('Negativpreise pro Monat (Solar-Überschuss-Indikator)', color='white')
    ax.legend(fontsize=8, framealpha=0.3, facecolor='#111', labelcolor='white')
    ax.grid(True, axis='y', alpha=0.10)
    ax.set_xlabel('Monat', color='#aaa')

    plt.tight_layout()
    p7 = os.path.join(CHARTS_DIR, 'kuer_nb08_spread_zeitreihe.png')
    plt.savefig(p7, dpi=DPI, bbox_inches='tight', facecolor='#0d1117')
    plt.show(); plt.close()
    trend_dir = 'steigend' if z[0] > 0 else 'fallend'
    print(f'Chart 7 gespeichert: {p7}')
    print(f'Trend: Spread {trend_dir} ({z[0]:+.3f} EUR/MWh/Monat = {z[0]*12:+.2f}/Jahr)')

    # ── Einzelplot 1: Intra-Tag-Spread mit Trend ──────────────────────────────
    for fname, col, title, ylabel, data_col, with_trend in [
        ('chart7a_spread_trend.png',  '#FFA726', 'Intra-Tag-Spread (p75−p25 Median)',
         'Spread [EUR/MWh]', 'spread_median', True),
        ('chart7b_volatilitaet.png',  '#42A5F5', 'Marktvolatilität (Std. Abweichung)',
         'Volatilität [EUR/MWh]', 'volatility', False),
    ]:
        fig_e, ax_e = plt.subplots(figsize=(12, 4))
        fig_e.patch.set_facecolor('#0d1117'); ax_e.set_facecolor('#141414')
        ax_e.tick_params(colors='#bbbbbb')
        for sp in ax_e.spines.values(): sp.set_edgecolor('#333')
        ax_e.fill_between(x, df_sp[data_col], alpha=0.3, color=col)
        ax_e.plot(x, df_sp[data_col], color=col, lw=2,
                  label=f'Ø {df_sp[data_col].mean():.1f} EUR/MWh')
        ax_e.axhline(df_sp[data_col].mean(), color='white', lw=1, linestyle='--', alpha=0.5)
        if with_trend:
            ax_e.plot(x, np.polyval(z, xn), color='#EF5350', lw=1.5, linestyle=':',
                      label=f'Trend ({z[0]:+.2f} EUR/MWh/Monat)')
        ax_e.set_title(title, color='white', fontsize=12)
        ax_e.set_ylabel(ylabel, color='#aaa')
        ax_e.set_xlabel('Monat', color='#aaa')
        ax_e.legend(fontsize=9, framealpha=0.3, facecolor='#111', labelcolor='white')
        ax_e.grid(True, alpha=0.10)
        plt.tight_layout()
        plt.savefig(os.path.join(CHARTS_DIR, fname), dpi=DPI, bbox_inches='tight', facecolor='#0d1117')
        plt.close(); print(f'  Einzelplot: {fname}')

    # ── Einzelplot 3: Negativpreise ───────────────────────────────────────────
    fig_e, ax_e = plt.subplots(figsize=(12, 4))
    fig_e.patch.set_facecolor('#0d1117'); ax_e.set_facecolor('#141414')
    ax_e.tick_params(colors='#bbbbbb')
    for sp in ax_e.spines.values(): sp.set_edgecolor('#333')
    colors_neg = ['#EF5350' if v > 0 else '#444' for v in df_sp['neg_price_h']]
    ax_e.bar(x, df_sp['neg_price_h'], color=colors_neg, alpha=0.85, width=25,
             label='Stunden mit Preis < 0 EUR/MWh')
    ax_e.set_title('Negativpreise pro Monat (Solar-Überschuss-Indikator)', color='white', fontsize=12)
    ax_e.set_ylabel('Negativpreis-Stunden', color='#aaa')
    ax_e.set_xlabel('Monat', color='#aaa')
    ax_e.legend(fontsize=9, framealpha=0.3, facecolor='#111', labelcolor='white')
    ax_e.grid(True, axis='y', alpha=0.10)
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS_DIR, 'kuer_nb08_negativpreise.png'),
                dpi=DPI, bbox_inches='tight', facecolor='#0d1117')
    plt.close(); print('  Einzelplot: kuer_nb08_negativpreise.png')

---
## 4. Analyse: CAPEX-Lernkurve

**Historische Entwicklung** (Quelle: BNEF / IEA, aus NB07_Erweiterungen Sektion 3):

| Jahr | CAPEX Cell [USD/kWh] | Systempreis CH (est.) |
|---|---|---|
| 2015 | ~350 | ~780 EUR/kWh |
| 2020 | ~140 | ~580 EUR/kWh |
| 2023 | ~90–130 | ~390–580 EUR/kWh |
| 2030 (Proj.) | ~60–80 | ~240–340 EUR/kWh |

*Systempreis = Cell + Wechselrichter + Installation + Montage (Faktor ~2–3×)*

### Projektion bei −10%/Jahr Lernrate

| Jahr | CAPEX Privat (Schätzung) | Break-Even bei Spread 25 EUR/MWh |
|---|---|---|
| 2025 | ~350 EUR/kWh | > 25 Jahre |
| 2028 | ~260 EUR/kWh | ~18 Jahre |
| 2031 | ~190 EUR/kWh | ~13 Jahre |
| 2034 | ~140 EUR/kWh | ~9 Jahre |

**Monitoring-Trigger:** Privat-Investition attraktiv wenn Systempreis  
**< 250 EUR/kWh installiert** — Schwellenwert in `config.json → kuer.markt.capex_ziel_privat_eur_kwh`.


In [ ]:
# ── CAPEX-Lernkurven-Projektion ───────────────────────────────────────────────
import datetime
basisjahr    = 2025
capex_basis  = 350   # EUR/kWh (Systempreis 2025, Schätzung)
lernrate     = CAPEX_LERNRATE / 100

jahre    = list(range(basisjahr, basisjahr + 16))
capex_pr = [capex_basis * (1 - lernrate)**i for i in range(len(jahre))]

print(f'CAPEX-Projektion ({CAPEX_LERNRATE:.0f}%/Jahr Lernrate):')
print(f'  {"Jahr":<6} {"CAPEX [EUR/kWh]":>16} {"Status":>30}')
for j, c in zip(jahre, capex_pr):
    status = '← TRIGGER UNTERSCHRITTEN' if c < CAPEX_TRIGGER and capex_pr[jahre.index(j)-1] >= CAPEX_TRIGGER else ''
    marker = '>>>' if status else '   '
    print(f'  {marker} {j:<4} {c:>14.0f}  {status}')


---
## 5. Chart 08-B: CAPEX × Spread Sensitivitäts-Heatmap

*Direkt übernommen aus NB03 Chart 8 — hier eingebettet für eigenständige Ausführbarkeit.*  
Break-Even-Jahre als Funktion von CAPEX [EUR/kWh] und Intra-Tag-Spread [EUR/MWh].  
Horizontale Linien = CAPEX-Lernkurve-Projektionsjahre; vertikale Linie = aktueller CH-Spread.


In [ ]:
# ── Chart 8: CAPEX × Spread Sensitivitäts-Heatmap ───────────────────────────
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# Parameter-Raum
capex_range  = np.arange(50, 501, 25)   # EUR/kWh
spread_range = np.arange(5, 101, 5)     # EUR/MWh Intra-Tag-Spread

# Batterieparameter Referenz: Privat 10kWh/5kW
CAPACITY_KWH  = 10
POWER_KW      = 5
EFFICIENCY    = CFG['pflicht']['simulation']['efficiency_roundtrip']
OPEX_RATE_C8  = CFG['pflicht']['wirtschaftlichkeit']['opex_rate']
LIFETIME_C8   = CFG['pflicht']['wirtschaftlichkeit']['lifetime_j']
DISPATCH_DAYS = 200  # geschätzte Dispatch-Tage/Jahr (aus Simulation)

# Break-Even Berechnung für alle CAPEX × Spread Kombinationen
BE_matrix = np.zeros((len(capex_range), len(spread_range)))
for ci, capex_kwh in enumerate(capex_range):
    capex_total = capex_kwh * CAPACITY_KWH
    opex_annual = capex_total * OPEX_RATE_C8
    for si, spread_mwh in enumerate(spread_range):
        gross = spread_mwh * POWER_KW * EFFICIENCY * DISPATCH_DAYS / 1000
        net   = gross - opex_annual
        BE_matrix[ci, si] = capex_total / net if net > 0 else 99
BE_plot = np.clip(BE_matrix, 0, 30)

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#141414')
ax.tick_params(colors='#bbbbbb')
for sp in ax.spines.values(): sp.set_edgecolor('#333333')

cmap = LinearSegmentedColormap.from_list('be',
    ['#66BB6A', '#FFA726', '#EF5350', '#4a0000'], N=256)
im = ax.imshow(BE_plot, aspect='auto', origin='lower', cmap=cmap,
               extent=[spread_range[0], spread_range[-1],
                       capex_range[0],  capex_range[-1]],
               vmin=0, vmax=30)
cbar = plt.colorbar(im, ax=ax, pad=0.02)
cbar.set_label('Break-Even [Jahre] (>25J = rot)', color='white')
cbar.ax.tick_params(colors='white')

# Konturlinien
CS = ax.contour(spread_range, capex_range, BE_plot,
                levels=[8, 12, 15, 20], colors='white', linewidths=1.2, alpha=0.8)
ax.clabel(CS, fmt='%dJ', colors='white', fontsize=9)

# Aktueller Spread aus spread_zeitreihe.csv (falls vorhanden)
SPREAD_FILE = os.path.join(DIR_INTER, 'spread_zeitreihe.csv')
current_spread = 20  # Fallback-Schätzung
if os.path.exists(SPREAD_FILE):
    _df_sp = pd.read_csv(SPREAD_FILE)
    current_spread = _df_sp['spread_median'].median()

# ── Marker: Heute (Spread × CAPEX) ────────────────────────────────────────────
today_spread = current_spread
today_capex  = 350
ax.plot(today_spread, today_capex,
        marker='*', color='white', markersize=18, zorder=10,
        label=f'Heute (~{today_spread:.0f} EUR/MWh, ~{today_capex} EUR/kWh)')
ax.annotate(f'Heute\n~{today_spread:.0f} EUR/MWh\n~{today_capex} EUR/kWh',
            xy=(today_spread, today_capex),
            xytext=(today_spread + 5, today_capex + 30),
            color='white', fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='white', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#222', alpha=0.7,
                      edgecolor='white', linewidth=1))

# ── Marker: Trigger ────────────────────────────────────────────────────────────
trig_spread = 30
trig_capex  = 250
ax.plot(trig_spread, trig_capex,
        marker='D', color='#66BB6A', markersize=11, zorder=10,
        label=f'Trigger ({trig_spread} EUR/MWh, {trig_capex} EUR/kWh)')
ax.annotate(f'Trigger\n{trig_spread} EUR/MWh\n{trig_capex} EUR/kWh',
            xy=(trig_spread, trig_capex),
            xytext=(trig_spread + 5, trig_capex - 60),
            color='#66BB6A', fontsize=8.5, fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#66BB6A', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#111', alpha=0.7,
                      edgecolor='#66BB6A', linewidth=1))

# ── CAPEX-Lernkurve ────────────────────────────────────────────────────────────
ax.axhline(today_capex, color='#42A5F5', lw=1.5, linestyle=':')
ax.text(spread_range[-1]*0.98, today_capex + 5,
        f'Heute ~{today_capex} EUR/kWh',
        color='#42A5F5', fontsize=8, ha='right')
for yrs, label in [(3, '-30% (3J)'), (5, '-50% (5J)')]:
    proj = today_capex * (0.90 ** yrs)
    ax.axhline(proj, color='#42A5F5', lw=1, linestyle=':', alpha=0.5)
    ax.text(spread_range[-1]*0.98, proj + 5, f'{label}: ~{proj:.0f} EUR/kWh',
            color='#42A5F5', fontsize=7.5, ha='right', alpha=0.8)

# ── Aktueller Spread (vertikale Linie) ────────────────────────────────────────
ax.axvline(today_spread, color='white', lw=1.5, linestyle='--', alpha=0.6)

ax.set_xlabel('Intra-Tag-Spread [EUR/MWh]', color='#aaa')
ax.set_ylabel('CAPEX [EUR/kWh]', color='#aaa')
ax.set_title(
    f'Break-Even-Jahre: CAPEX \u00d7 Spread-Sensitivit\u00e4t'
    f' (Privat {CAPACITY_KWH}kWh/{POWER_KW}kW, {DISPATCH_DAYS} Dispatch-Tage/J)\n'
    'Gr\u00fcn = kurz amortisiert \u00b7 Rot = nie \u00b7 Konturlinien = Break-Even Jahre',
    color='white', fontsize=11)
ax.legend(fontsize=8.5, framealpha=0.5, facecolor='#111',
          labelcolor='white', loc='lower right')

plt.tight_layout()
p8 = os.path.join(CHARTS_DIR, 'kuer_nb08_sensitivitaet_heatmap.png')
plt.savefig(p8, dpi=DPI, bbox_inches='tight', facecolor='#0d1117')
plt.show(); plt.close()
print(f'Chart 8 gespeichert: {p8}')

---
## 6. Synthese: Wann kippt der Business Case?

Beide Parameter entwickeln sich in die richtige Richtung — aber sie müssen  
**gleichzeitig** einen Schwellenwert erreichen. Einzeln reicht keiner.

| Faktor | Heute | Trigger | Zeithorizont |
|--------|-------|---------|--------------|
| Spread | Ø aus spread_zeitreihe.csv | > 30 EUR/MWh | 2–5 Jahre |
| CAPEX  | ~350 EUR/kWh | < 250 EUR/kWh | 3–6 Jahre |
| Beide zusammen | — | → Break-Even ~10 Jahre | ~2028–2030 |

> Vollständige Trigger-Matrix mit weiteren Parametern (VPP, DA-Dispatch, BVI) → **NB05 Business Strategy**.

---
| [← NB07 Cross-Border](07_Cross_Border.ipynb) | [↑ Projektübersicht](00_Project_Overview.ipynb) | [→ NB09 Revenue Stacking](09_Revenue_Stacking.ipynb) |
|:---|:---:|---:|

### Technologie-Sensitivität

Die CAPEX-Heatmap zeigt: bei **~180 EUR/kWh** (aktueller Marktpreis für LFP-Grossbatterien, z.B. 10 kWh für 1'800 EUR) verkürzt sich der Break-Even bei einem Spread von 30 EUR/MWh auf unter 10 Jahre — der Business Case dreht sich.

| Technologie | CAPEX heute | Lebensdauer | Besonderheit |
|-------------|------------|-------------|----------|
| **LFP** | ~180–220 EUR/kWh | 15–20 J, 4'000+ Zyklen | Günstig, langlebig, aktuell dominierend |
| **NMC** *(modelliert)* | 300–400 EUR/kWh | 10–12 J, ~1'500 Zyklen | Höhere Energiedichte |
| **Redox-Flow** | ~400–600 EUR/kWh | 20+ J, unbegrenzt | Ideal für stationäre Grossspeicher |
| **CAES** | ~50–100 EUR/kWh | 30+ J | Geografisch gebunden, andere Regulierung |

> Vollständiger Technologievergleich inkl. Wirtschaftlichkeit: **NB11 Technologievergleich** *(geplant)*.
